# Tri-Stream Pack Control Panel v0.1

Opt-in pack stage for `training-data-v4-tri-stream`. Run detect and silhouette first.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if (PROJECT_ROOT / '02_synthetic-data-processing-v4.0' / 'rb_pipeline_v4').exists():
    PROJECT_ROOT = PROJECT_ROOT / '02_synthetic-data-processing-v4.0'
else:
    for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
        if (candidate / 'rb_pipeline_v4').exists():
            PROJECT_ROOT = candidate
            break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import ipywidgets as widgets
from IPython.display import clear_output, display
import matplotlib.pyplot as plt

from rb_pipeline_v4.config import BrightnessNormalizationConfigV4
from rb_pipeline_v4.tri_stream_control import (
    build_pack_tri_stream_config,
    discover_tri_stream_runs,
    infer_tri_stream_run_canvas_size,
    preview_tri_stream_sample,
    run_tri_stream_pack,
    tri_stream_output_root,
)

output_root_label = widgets.HTML()
run_selector = widgets.Dropdown(description='Input run', options=[])
refresh_button = widgets.Button(description='Refresh runs', button_style='')
canvas_width = widgets.IntText(description='Canvas W', value=224)
canvas_height = widgets.IntText(description='Canvas H', value=224)
clip_policy = widgets.Dropdown(description='Clip', options=['fail', 'clip'], value='fail')
orientation_context_scale = widgets.FloatText(description='Orient scale', value=1.25)
brightness_enabled = widgets.Checkbox(description='Brightness norm', value=False)
brightness_target = widgets.FloatText(description='Target darkness', value=0.55)
shard_size = widgets.IntText(description='Shard size', value=8192)
overwrite = widgets.Checkbox(description='Overwrite', value=False)
sample_offset = widgets.IntText(description='Offset', value=0)
sample_limit = widgets.IntText(description='Limit', value=0)
preview_index = widgets.IntText(description='Preview row', value=0)
preview_button = widgets.Button(description='Preview', button_style='info')
run_button = widgets.Button(description='Run pack', button_style='warning')
status_panel = widgets.HTML()
log_panel = widgets.Output(layout={'border': '1px solid #ddd', 'max_height': '240px', 'overflow': 'auto'})
preview_panel = widgets.Output()

def _brightness_config():
    return BrightnessNormalizationConfigV4(
        enabled=bool(brightness_enabled.value),
        method='masked_median_darkness_gain' if brightness_enabled.value else 'none',
        target_median_darkness=float(brightness_target.value),
    )

def _pack_config():
    return build_pack_tri_stream_config(
        canvas_width_px=int(canvas_width.value),
        canvas_height_px=int(canvas_height.value),
        clip_policy=str(clip_policy.value),
        orientation_context_scale=float(orientation_context_scale.value),
        shard_size=int(shard_size.value),
        overwrite=bool(overwrite.value),
        sample_offset=int(sample_offset.value),
        sample_limit=int(sample_limit.value),
        brightness_normalization=_brightness_config(),
    )

def _sync_canvas_to_selected_run(_=None):
    if not run_selector.value:
        return
    try:
        width, height = infer_tri_stream_run_canvas_size(PROJECT_ROOT, str(run_selector.value))
        canvas_width.value = int(width)
        canvas_height.value = int(height)
    except Exception as exc:
        status_panel.value = f"Could not infer silhouette canvas for <code>{run_selector.value}</code>: {exc}"

def _refresh_runs(_=None):
    output_root_label.value = f"Output root: <code>{tri_stream_output_root(PROJECT_ROOT)}</code>"
    statuses = discover_tri_stream_runs(PROJECT_ROOT)
    options = []
    for item in statuses:
        label = item.run_name
        if item.row_count is not None:
            label = f"{label} ({item.row_count} rows)"
        if item.output_exists:
            label = f"{label} [output exists]"
        if item.issues:
            label = f"{label} [not ready]"
        options.append((label, item.run_name))
    run_selector.options = options
    status_panel.value = f"Discovered {len(options)} input run(s)."
    _sync_canvas_to_selected_run()

def _show_preview(_=None):
    with preview_panel:
        clear_output(wait=True)
        result = preview_tri_stream_sample(
            PROJECT_ROOT,
            str(run_selector.value),
            _pack_config(),
            row_index=int(preview_index.value),
        )
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))
        axes[0].imshow(result.arrays['source_roi_canvas'], cmap='gray', vmin=0, vmax=255)
        axes[0].set_title('source ROI canvas')
        axes[1].imshow(result.arrays['x_distance_image'], cmap='gray', vmin=0, vmax=1)
        axes[1].set_title('x_distance_image')
        axes[2].imshow(result.arrays['x_orientation_image'], cmap='gray', vmin=0, vmax=1)
        axes[2].set_title('x_orientation_image')
        for axis in axes:
            axis.axis('off')
        plt.show()
        display(result.geometry_summary)
        status_panel.value = f"Previewed row {result.row_index} / sample <code>{result.sample_id}</code>."

def _append_log(message: str):
    with log_panel:
        print(message)

def _run_pack(_=None):
    with log_panel:
        clear_output(wait=True)
    summary = run_tri_stream_pack(
        PROJECT_ROOT,
        str(run_selector.value),
        _pack_config(),
        log_sink=_append_log,
    )
    status_panel.value = (
        f"pack_tri_stream complete: success={summary.successful_rows}, "
        f"failed={summary.failed_rows}, skipped={summary.skipped_rows}; "
        f"output=<code>{summary.output_path}</code>"
    )

refresh_button.on_click(_refresh_runs)
run_selector.observe(_sync_canvas_to_selected_run, names='value')
preview_button.on_click(_show_preview)
run_button.on_click(_run_pack)
_refresh_runs()

controls = widgets.VBox([
    output_root_label,
    widgets.HBox([run_selector, refresh_button]),
    widgets.HBox([canvas_width, canvas_height, clip_policy, orientation_context_scale]),
    widgets.HBox([brightness_enabled, brightness_target]),
    widgets.HBox([shard_size, overwrite, sample_offset, sample_limit]),
    widgets.HBox([preview_index, preview_button, run_button]),
    status_panel,
])
display(widgets.VBox([controls, preview_panel, log_panel]))